# Clustering Comparison: Ours vs KDSC vs TDBM (AV2 val)

Side-by-side comparison of three driving-style labellings on the same AV2 val rows:

| method | label column        | source                                                            |
|--------|---------------------|-------------------------------------------------------------------|
| ours   | `style_label`       | `artifacts/clustering/av2_val_clustered.parquet`                  |
| KDSC   | `style_label_kdsc`  | `artifacts/kdsc_replication/av2/av2_val_kdsc.parquet`             |
| TDBM   | `style_label_tdbm`  | `artifacts/tdbm_replication/av2/av2_val_tdbm.parquet` (binary)    |
| TDBM-6 | `tdbm_raw_style_label` | same file (6-way raw)                                         |

We inner-join the three on `(scenario_id, center_objects_id)` so every comparison row has all three labels.

**Bias disclosure.** The inner join intersects three different cleaning policies, so the comparison set is whichever rows survive *all* of them - in practice limited by our (heavier) stop-and-go + physical-sanity filter. We don't try to hide this: every reported metric is on the same row set, but readers should know that row selection is **not** neutral between methods.

Outputs go to `artifacts/comparison/av2/`.

In [1]:
from __future__ import annotations
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATASET = 'av2'

_CWD = Path.cwd()
for cand in (_CWD, _CWD.parent, _CWD.parent.parent):
    if (cand / 'src').is_dir():
        REPO_ROOT = cand
        break
else:
    REPO_ROOT = _CWD
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

OURS_PARQUET = REPO_ROOT / 'artifacts' / 'clustering' / f'{DATASET}_val_clustered.parquet'
KDSC_PARQUET = REPO_ROOT / 'artifacts' / 'kdsc_replication' / DATASET / f'{DATASET}_val_kdsc.parquet'
TDBM_PARQUET = REPO_ROOT / 'artifacts' / 'tdbm_replication' / DATASET / f'{DATASET}_val_tdbm.parquet'

OUT_DIR = REPO_ROOT / 'artifacts' / 'comparison' / DATASET
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

for name, p in [('ours', OURS_PARQUET), ('kdsc', KDSC_PARQUET), ('tdbm', TDBM_PARQUET)]:
    if not p.exists():
        raise FileNotFoundError(f'Missing {name} parquet: {p}')
    print(f'{name}: {p}')

ours: /fs/nexus-projects/pc_driving/yaghoubi/tail-risk-motion-prediction/artifacts/clustering/av2_val_clustered.parquet
kdsc: /fs/nexus-projects/pc_driving/yaghoubi/tail-risk-motion-prediction/artifacts/kdsc_replication/av2/av2_val_kdsc.parquet
tdbm: /fs/nexus-projects/pc_driving/yaghoubi/tail-risk-motion-prediction/artifacts/tdbm_replication/av2/av2_val_tdbm.parquet


## 1. Load and join the three label sets

In [2]:
KEY = ['scenario_id', 'center_objects_id']

def _norm_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['scenario_id']       = out['scenario_id'].astype(str)
    out['center_objects_id'] = out['center_objects_id'].astype(str)
    return out

ours = _norm_keys(pd.read_parquet(OURS_PARQUET))[KEY + ['style_label']]
kdsc = _norm_keys(pd.read_parquet(KDSC_PARQUET))[KEY + ['style_label_kdsc']]
tdbm = _norm_keys(pd.read_parquet(TDBM_PARQUET))[KEY + [
    'style_label_tdbm', 'tdbm_raw_style_id', 'tdbm_raw_style_label', 'tdbm_has_neighbor']]

# We also want the raw kinematic columns from one of the parquets (TDBM keeps the most).
tdbm_full = _norm_keys(pd.read_parquet(TDBM_PARQUET))

joined = (ours
    .merge(kdsc, on=KEY, how='inner')
    .merge(tdbm, on=KEY, how='inner')
    .merge(tdbm_full.drop(columns=['style_label_tdbm', 'tdbm_raw_style_id',
                                    'tdbm_raw_style_label', 'tdbm_has_neighbor']),
           on=KEY, how='inner'))

print('Per-source sizes:')
print(f'  ours: {len(ours):>6}')
print(f'  kdsc: {len(kdsc):>6}')
print(f'  tdbm: {len(tdbm):>6}')
print(f'  joined (inner): {len(joined)}')
joined.head(3)

Per-source sizes:
  ours:  12387
  kdsc:  21809
  tdbm:  21809
  joined (inner): 12387


,scenario_id,center_objects_id,style_label,style_label_kdsc,style_label_tdbm,tdbm_raw_style_id,tdbm_raw_style_label,tdbm_has_neighbor,dataset,split,...,tdbm_v_nei,tdbm_s_front,tdbm_v_avg,tdbm_j_l,tdbm_score_0,tdbm_score_1,tdbm_score_2,tdbm_score_3,tdbm_score_4,tdbm_score_5
0,00010486-9a07-48ae-b493-cf4545855937,77544,normal,normal,normal,3,Careful,True,av2,val,...,-96.482697,5.306903,8.848875,6.449143,-396.392090,-301.692749,-404.328156,314.089355,263.097473,234.567993
1,00062a32-8d6d-4449-9948-6fedac67bfcd,21361,aggressive,normal,normal,3,Careful,True,av2,val,...,-859.744568,3.606738,9.380055,10.875275,-3475.728516,-2652.206299,-3518.918213,2729.841064,2242.038086,1902.727417
2,0006ca28-fcbb-4ae2-9d9e-951fa3b41c1c,62021,normal,normal,normal,3,Careful,True,av2,val,...,-650.307312,18.619064,11.072953,8.787586,-2639.732910,-2013.652344,-2673.264648,2083.765869,1722.339844,1473.886475


## 2. Per-method aggressive share

Sanity check: do the three methods even agree on what fraction of agents look aggressive?

In [3]:
share = pd.DataFrame({
    'method': ['ours', 'kdsc', 'tdbm'],
    'aggressive_share': [
        float((joined['style_label']      == 'aggressive').mean()),
        float((joined['style_label_kdsc'] == 'aggressive').mean()),
        float((joined['style_label_tdbm'] == 'aggressive').mean()),
    ],
    'aggressive_count': [
        int((joined['style_label']      == 'aggressive').sum()),
        int((joined['style_label_kdsc'] == 'aggressive').sum()),
        int((joined['style_label_tdbm'] == 'aggressive').sum()),
    ],
    'total': len(joined),
})
share.to_csv(OUT_DIR / 'aggressive_share.csv', index=False)
share

,method,aggressive_share,aggressive_count,total
0,ours,0.183176,2269,12387
1,kdsc,0.000000,0,12387
2,tdbm,0.000081,1,12387


## 3. Pairwise agreement (binary labels)

Three pairs: ours vs KDSC, ours vs TDBM, KDSC vs TDBM. For each we report

- the 2x2 confusion matrix,
- raw label-overlap accuracy,
- Cohen's kappa (chance-corrected agreement),
- Jaccard on the `aggressive` class (how much the two `aggressive` sets actually overlap).

In [4]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix, jaccard_score

def _binary_agreement(a: pd.Series, b: pd.Series) -> dict:
    cm = confusion_matrix(a, b, labels=['aggressive', 'normal'])
    acc = float((a == b).mean())
    kappa = float(cohen_kappa_score(a, b))
    jaccard = float(jaccard_score(a == 'aggressive', b == 'aggressive'))
    return {'confusion': cm, 'accuracy': acc, 'cohen_kappa': kappa, 'jaccard_aggressive': jaccard}

pairs = {
    'ours_vs_kdsc': ('style_label',      'style_label_kdsc'),
    'ours_vs_tdbm': ('style_label',      'style_label_tdbm'),
    'kdsc_vs_tdbm': ('style_label_kdsc', 'style_label_tdbm'),
}
results: dict[str, dict] = {}
for name, (a, b) in pairs.items():
    results[name] = _binary_agreement(joined[a], joined[b])

summary = pd.DataFrame({
    name: {'accuracy': r['accuracy'], 'cohen_kappa': r['cohen_kappa'],
           'jaccard_aggressive': r['jaccard_aggressive']}
    for name, r in results.items()
}).T.round(3)
summary.to_csv(OUT_DIR / 'pairwise_agreement.csv')
summary

,accuracy,cohen_kappa,jaccard_aggressive
ours_vs_kdsc,0.817,0.000,0.0
ours_vs_tdbm,0.817,0.001,0.0
kdsc_vs_tdbm,1.000,0.000,0.0


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, (a, b)) in zip(axes, pairs.items()):
    cm = results[name]['confusion']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['aggressive', 'normal'],
                yticklabels=['aggressive', 'normal'], cbar=False)
    left, right = name.split('_vs_')
    ax.set_xlabel(right); ax.set_ylabel(left)
    ax.set_title(f'{name}\nkappa={results[name]["cohen_kappa"]:.2f}')
fig.tight_layout()
fig.savefig(FIG_DIR / 'pairwise_confusion.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Per-cluster kinematic profile (z-score, fixed feature set)

For each method we compute the standardized mean of a fixed set of kinematic features per cluster. Same features, same rows, three label columns - so the heatmaps are directly comparable. The 'aggressive' column should have positive z-scores on aggressive-leaning features (lateral-G spikes, hard braking, sharp heading, jerk) and not just on raw speed.

In [ ]:
PROFILE_FEATURES = [c for c in [
    'avg_speed', 'max_speed', 'var_speed',
    'max_abs_accel', 'var_acceleration', 'gamma',
    'max_abs_long_accel', 'max_abs_lat_accel',
    'max_abs_jerk', 'max_abs_long_jerk', 'max_abs_lat_jerk',
    'hard_brake_count', 'hard_accel_count',
    'lateral_g_spike_count', 'high_jerk_count',
    'max_abs_heading_rate',
] if c in joined.columns]
print('profile features:', PROFILE_FEATURES)

z = joined[PROFILE_FEATURES].astype(float)
z = (z - z.mean()) / z.std()

method_label_cols = {
    'ours': 'style_label',
    'kdsc': 'style_label_kdsc',
    'tdbm': 'style_label_tdbm',
}
profiles = {}
for method, lab_col in method_label_cols.items():
    p = z.assign(_lab=joined[lab_col]).groupby('_lab')[PROFILE_FEATURES].mean().T
    p = p[[c for c in ['aggressive', 'normal'] if c in p.columns]]
    profiles[method] = p

fig, axes = plt.subplots(1, 3, figsize=(13, max(4, 0.32 * len(PROFILE_FEATURES))),
                          sharey=True)
for ax, method in zip(axes, method_label_cols):
    sns.heatmap(profiles[method], annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                ax=ax, cbar=ax is axes[-1], cbar_kws={'label': 'z-score'})
    ax.set_title(method.upper())
    ax.set_xlabel('')
fig.suptitle(f'AV2 val: per-cluster z-score (n={len(joined)})')
fig.tight_layout()
fig.savefig(FIG_DIR / 'cluster_profiles_side_by_side.png', dpi=120, bbox_inches='tight')
plt.show()

# also save the wide CSV
wide = pd.concat({m: p for m, p in profiles.items()}, axis=1)
wide.to_csv(OUT_DIR / 'cluster_profiles.csv')

In [ ]:
# Single bar plot: aggressive-vs-normal delta on key features, per method.
delta_rows = []
for method, p in profiles.items():
    if {'aggressive', 'normal'}.issubset(p.columns):
        delta = (p['aggressive'] - p['normal']).rename(method)
        delta_rows.append(delta)
delta_df = pd.concat(delta_rows, axis=1) if delta_rows else None

if delta_df is not None:
    delta_df = delta_df.sort_values('ours', ascending=True)
    ax = delta_df.plot.barh(figsize=(9, max(4, 0.35 * len(delta_df))),
                            width=0.8, edgecolor='none')
    ax.axvline(0, color='black', linewidth=0.6)
    ax.set_xlabel('z-score(aggressive) - z-score(normal)')
    ax.set_title('Per-feature aggressive-vs-normal gap (larger = method calls feature "aggressive")')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'aggressive_normal_delta.png', dpi=120, bbox_inches='tight')
    plt.show()
    delta_df.to_csv(OUT_DIR / 'aggressive_normal_delta.csv')
    delta_df.round(3)

## 5. Internal metrics in two shared feature spaces

Each method optimizes its own feature space, so silhouette / DB / CH on *that* space is biased toward whoever picked it. To stay honest we report internal metrics in **two** common standardized spaces:

- **KDSC space** (paper's 4 features): `max_abs_accel`, `var_acceleration`, `var_speed`, `gamma`
- **Our space** (8 lateral/braking features used by `kmeans_K2.pkl`)

Reporting both lets a reader see whose feature choice each method is best aligned with.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import pickle

with open(REPO_ROOT / 'artifacts' / 'clustering' / 'kmeans_K2.pkl', 'rb') as f:
    ours_bundle = pickle.load(f)
OUR_FEATURES = ours_bundle['features']
KDSC_FEATURES_4 = ['max_abs_accel', 'var_acceleration', 'var_speed', 'gamma']

def _internal(X: np.ndarray, labels: np.ndarray, max_silhouette_sample: int = 8000) -> dict:
    if len(set(labels)) < 2:
        return {'silhouette': float('nan'),
                'davies_bouldin': float('nan'),
                'calinski_harabasz': float('nan')}
    rng = np.random.default_rng(42)
    sub = rng.choice(len(X), size=min(max_silhouette_sample, len(X)), replace=False)
    return {
        'silhouette':        float(silhouette_score(X[sub], labels[sub])),
        'davies_bouldin':    float(davies_bouldin_score(X, labels)),
        'calinski_harabasz': float(calinski_harabasz_score(X, labels)),
    }

spaces = {
    'kdsc_4feat': [c for c in KDSC_FEATURES_4 if c in joined.columns],
    'ours_8feat': [c for c in OUR_FEATURES   if c in joined.columns],
}

rows = []
for space, feats in spaces.items():
    if len(feats) == 0:
        continue
    X = joined[feats].astype(float).to_numpy()
    Xs = StandardScaler().fit_transform(X)
    for method, lab_col in method_label_cols.items():
        labels = joined[lab_col].to_numpy()
        m = _internal(Xs, labels)
        m.update({'space': space, 'method': method, 'n_features': len(feats)})
        rows.append(m)
metrics_df = pd.DataFrame(rows)[['space', 'method', 'n_features',
                                  'silhouette', 'davies_bouldin', 'calinski_harabasz']]
metrics_df.to_csv(OUT_DIR / 'internal_metrics.csv', index=False)
metrics_df.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ['silhouette', 'davies_bouldin', 'calinski_harabasz']):
    sns.barplot(metrics_df, x='space', y=metric, hue='method', ax=ax)
    ax.set_title(metric)
    if metric == 'davies_bouldin':
        ax.set_ylabel(metric + '  (lower = better)')
    else:
        ax.set_ylabel(metric + '  (higher = better)')
    ax.tick_params(axis='x', rotation=10)
fig.suptitle('Internal cluster metrics in two shared feature spaces')
fig.tight_layout()
fig.savefig(FIG_DIR / 'internal_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Disagreement manifest

Save lists of rows where methods disagree, so we can hand-pick disagreement cases and render them with one of the GIF notebooks. Two flavors:

- **ours-only-aggressive**: we call it aggressive, the baseline does not.
- **baseline-only-aggressive**: the baseline calls it aggressive, we don't.

Picking these by `extremeness` / `tdbm_margin` is the natural way to find the most-decisive disagreements (also saved to the CSV for downstream filtering).

In [ ]:
# Need extremeness from our parquet; re-load only the column we want and merge in.
ours_full = _norm_keys(pd.read_parquet(OURS_PARQUET))
if 'extremeness' not in joined.columns and 'extremeness' in ours_full.columns:
    joined = joined.merge(ours_full[KEY + ['extremeness']], on=KEY, how='left')

tdbm_score_cols = [c for c in joined.columns if c.startswith('tdbm_score_')]
if 'tdbm_margin' not in joined.columns and tdbm_score_cols:
    sc = joined[tdbm_score_cols].to_numpy(dtype=np.float32)
    own = sc[np.arange(len(sc)), joined['tdbm_raw_style_id'].to_numpy(dtype=np.int64)]
    masked = sc.copy()
    masked[np.arange(len(sc)), joined['tdbm_raw_style_id'].to_numpy(dtype=np.int64)] = -np.inf
    joined['tdbm_margin'] = own - masked.max(axis=1)

DISPLAY_COLS = [c for c in [
    'scenario_id', 'center_objects_id',
    'style_label', 'style_label_kdsc', 'style_label_tdbm', 'tdbm_raw_style_label',
    'extremeness', 'tdbm_margin',
    'avg_speed', 'max_abs_accel', 'max_abs_lat_accel',
    'hard_brake_count', 'lateral_g_spike_count',
] if c in joined.columns]

def _save_pair(name: str, mask: pd.Series, sort_by: str):
    sub = joined.loc[mask, DISPLAY_COLS]
    if sort_by in sub.columns:
        sub = sub.sort_values(sort_by, ascending=False)
    out_csv = OUT_DIR / f'disagreement_{name}.csv'
    sub.to_csv(out_csv, index=False)
    print(f'{name:40s}  rows={len(sub):>6}  -> {out_csv.relative_to(REPO_ROOT)}')
    return sub

print('Disagreement counts and CSVs:')
ours_agg_only_vs_kdsc = _save_pair('ours_aggressive_kdsc_normal',
    (joined['style_label'] == 'aggressive') & (joined['style_label_kdsc'] == 'normal'),
    sort_by='extremeness')
kdsc_agg_only_vs_ours = _save_pair('kdsc_aggressive_ours_normal',
    (joined['style_label_kdsc'] == 'aggressive') & (joined['style_label'] == 'normal'),
    sort_by='extremeness')
ours_agg_only_vs_tdbm = _save_pair('ours_aggressive_tdbm_normal',
    (joined['style_label'] == 'aggressive') & (joined['style_label_tdbm'] == 'normal'),
    sort_by='extremeness')
tdbm_agg_only_vs_ours = _save_pair('tdbm_aggressive_ours_normal',
    (joined['style_label_tdbm'] == 'aggressive') & (joined['style_label'] == 'normal'),
    sort_by='tdbm_margin')

## 7. Three-way label distribution

Crosstab over the full (ours, KDSC, TDBM) triple. Cells with high counts are 'all three agree'; cells off the all-agree diagonal are where the methods disagree.

In [ ]:
tri = (joined.groupby(['style_label', 'style_label_kdsc', 'style_label_tdbm'])
              .size()
              .rename('count')
              .reset_index()
              .sort_values('count', ascending=False))
tri['fraction'] = tri['count'] / len(joined)
tri.to_csv(OUT_DIR / 'three_way_distribution.csv', index=False)
tri.round(3)

In [ ]:
# How does TDBM's 6-way raw breakdown distribute across our binary labels?
tdbm_raw_x_ours = pd.crosstab(joined['tdbm_raw_style_label'],
                               joined['style_label'],
                               margins=True, margins_name='total')
tdbm_raw_x_ours.to_csv(OUT_DIR / 'tdbm_raw_vs_ours.csv')
tdbm_raw_x_ours

## 8. Final report

In [ ]:
report = {
    'dataset': DATASET,
    'n_joined': int(len(joined)),
    'aggressive_share': share.set_index('method')['aggressive_share'].to_dict(),
    'pairwise_agreement': {k: {'accuracy': v['accuracy'],
                                'cohen_kappa': v['cohen_kappa'],
                                'jaccard_aggressive': v['jaccard_aggressive']}
                           for k, v in results.items()},
    'internal_metrics': metrics_df.to_dict(orient='records'),
    'profile_features_used': PROFILE_FEATURES,
    'tdbm_no_neighbor_default_share': float((~joined['tdbm_has_neighbor']).mean()),
    'note': ('Inner join uses our cleaning pipeline\'s row set, so all metrics '
             'are computed on that subset. Report this explicitly with any results.'),
}
with open(OUT_DIR / 'comparison_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))
print('\nWrote:')
for p in sorted(OUT_DIR.rglob('*')):
    if p.is_file():
        print(' ', p.relative_to(REPO_ROOT))